In [1]:
import json

kaggle_dict = {
    "username": "srushtihg",
    "key": "KGAT_f9cb1ca1640233af667c3bc242f48f59"
}

with open("kaggle.json", "w") as f:
    json.dump(kaggle_dict, f)

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d mlg-ulb/creditcardfraud

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:02<00:00, 24.0MB/s]



In [4]:
!unzip creditcardfraud.zip

Archive:  creditcardfraud.zip
  inflating: creditcard.csv          


In [5]:
import pandas as pd

df = pd.read_csv("creditcard.csv")
print(df.shape)

(284807, 31)


In [6]:
print(df.shape)
print(df['Class'].value_counts())

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


In [7]:
df = df.sample(n=30000, random_state=42)

In [8]:
X = df.drop('Class', axis=1)
y = df['Class']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
X_train_normal = X_train_scaled[y_train == 0]

In [12]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_train_normal.shape[1]

input_layer = Input(shape=(input_dim,))

# Smaller = faster
encoded = Dense(8, activation="relu")(input_layer)
encoded = Dense(4, activation="relu")(encoded)

decoded = Dense(8, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

autoencoder.compile(optimizer='adam', loss='mse')

In [13]:
autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

Epoch 1/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.9405 - val_loss: 0.9945
Epoch 2/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8915 - val_loss: 0.9285
Epoch 3/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8374 - val_loss: 0.8796
Epoch 4/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8009 - val_loss: 0.8492
Epoch 5/5
169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.7790 - val_loss: 0.8342


In [14]:
import numpy as np

train_recon = autoencoder.predict(X_train_scaled)
test_recon = autoencoder.predict(X_test_scaled)

train_error = np.mean((X_train_scaled - train_recon)**2, axis=1)
test_error = np.mean((X_test_scaled - test_recon)**2, axis=1)

750/750 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [15]:
X_train_new = np.hstack((X_train_scaled, train_error.reshape(-1,1)))
X_test_new = np.hstack((X_test_scaled, test_error.reshape(-1,1)))

In [16]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(random_state=42)
X_res, y_res = adasyn.fit_resample(X_train_new, y_train)

In [17]:
from xgboost import XGBClassifier

# Calculate imbalance weight
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_res, y_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [18]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, f1_score
import numpy as np

y_prob = model.predict_proba(X_test_new)[:,1]

# 🔥 Find best threshold automatically
thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = 0
best_f1 = 0

for t in thresholds:
    y_pred_temp = (y_prob > t).astype(int)
    score = f1_score(y_test, y_pred_temp)

    if score > best_f1:
        best_f1 = score
        best_thresh = t

print("Best Threshold:", best_thresh)

# ✅ Final prediction using best threshold
y_pred = (y_prob > best_thresh).astype(int)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Best Threshold: 0.7500000000000002

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5991
           1       0.67      0.67      0.67         9

    accuracy                           1.00      6000
   macro avg       0.83      0.83      0.83      6000
weighted avg       1.00      1.00      1.00      6000

ROC-AUC: 0.988742372818487

Confusion Matrix:

[[5988    3]
 [   3    6]]
